In [1]:
import os
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import duckdb

Read each csv in the data folder and use seaborn to create / analyse stats about categorical data and dates found

1) Review acq_orders

In [ ]:

# 1) acq_orders.csv
acq_orders = pd.read_csv('../data/acq_orders.csv')

# print number of records in dataset
print(f"Number of records in acq_orders: {len(acq_orders)}")

display(acq_orders.head(5))
# explore cardinality of taxonomy_business_category_group
print(acq_orders['taxonomy_business_category_group'].value_counts())

# check if any changes in taxonomy_business_category_group per customer_id
customer_category_changes = acq_orders.groupby('customer_id')['taxonomy_business_category_group'].nunique()

#see if multiple groups per user (FALSE)
print(f"Number of customers with category changes: {customer_category_changes[customer_category_changes > 1].count()}")


Number of records in acq_orders: 508694


,customer_id,taxonomy_business_category_group
0,1026819,Weight Loss Group
1,1169512,Weight Loss Group
2,1204351,Weight Loss Group
3,1019780,Weight Loss Group
4,1166196,Weight Loss Group


taxonomy_business_category_group
Hair Loss Group        383273
ED Group                73086
Weight Loss Group       41736
Other Group              7038
Sleep Group              2275
TRT Group                1285
Mental Health Group         1
Name: count, dtype: int64
Number of customers with category changes: 0


^^ NOTE: Only a single record for Mental Health Group

2) Review activity.csv

In [3]:
# 2) activity.csv
activity = pd.read_csv('../data/activity.csv')
print(f"Number of records in activity.csv: {len(activity)}")
# print median number of records per customer_id
median_records_per_customer = activity.groupby('customer_id').size().median()
print(f"Median number of records per customer_id: {median_records_per_customer}")

display(activity.head(5))

#check if any to_date <= from_date
activity['from_date'] = pd.to_datetime(activity['from_date'])
activity['to_date'] = pd.to_datetime(activity['to_date'])
invalid_dates = activity[activity['to_date'] < activity['from_date']]
print(f"Number of records with invalid dates: {len(invalid_dates)}")


# check for changes in subscription attributes
# Method 1: Check if count equals unique count
is_unique = activity['subscription_id'].nunique() == len(activity)
print(f"Is subscription_id unique? {is_unique}")

# Method 2: Check for duplicates
has_duplicates = activity['subscription_id'].duplicated().any()
print(f"Has duplicates? {has_duplicates}")

# Method 3: See duplicate counts
duplicate_count = activity['subscription_id'].duplicated().sum()
print(f"Number of duplicates: {duplicate_count}")

# Method 4: Find which subscription_ids are duplicated
if has_duplicates:
    duplicated_subs = activity[activity['subscription_id'].duplicated(keep=False)]
    print("\nDuplicated subscription_ids:")
    print(duplicated_subs.sort_values('subscription_id'))

Number of records in activity.csv: 2176168
Median number of records per customer_id: 3.0


,customer_id,subscription_id,from_date,to_date
0,31922,19621,2019-09-06,2019-10-07
1,3095,36471,2020-05-26,2020-06-19
2,74237,54966,2020-11-02,2021-03-02
3,29966,60229,2020-11-30,2021-03-02
4,64520,46879,2020-12-04,2021-03-02


Number of records with invalid dates: 0
Is subscription_id unique? False
Has duplicates? True
Number of duplicates: 694166

Duplicated subscription_ids:
         customer_id  subscription_id  from_date    to_date
2139655           49               16 2019-01-15 2022-08-23
2159558           49               16 2022-08-25 2024-07-31
1648530           49               16 2024-08-01 2024-08-06
969482           326               73 2023-11-16 2023-11-20
2007932          326               73 2021-05-24 2023-11-10
...              ...              ...        ...        ...
1615636       203216          1761704 2024-08-15 2024-08-15
2174783       427167          1762204 2024-08-16 2024-08-16
1654831       427167          1762204 2024-08-15 2024-08-16
1220259       427167          1762205 2024-08-16 2024-08-16
2098949       427167          1762205 2024-08-15 2024-08-16

[1010500 rows x 4 columns]


Above, we can see evolution of subscription dates, though we don't know for which product

In [21]:
# print number of records in customers.csv
customers = pd.read_csv('../data/customers.csv')
display(customers.head(5))

# check cardinality of customer_country
print(customers['customer_country'].value_counts())

customer_country_counts = customers.groupby('customer_id')['customer_country'].nunique()
if (customer_country_counts > 1).any():
    print("There are customers with multiple countries.")


,customer_id,customer_country
0,201441,United Kingdom
1,122538,United Kingdom
2,38482,United Kingdom
3,7792,United Kingdom
4,289880,United Kingdom


customer_country
Brazil            296693
United Kingdom    236155
Name: count, dtype: int64


In [ ]:
# use duck db to read  ../data/activity and query for records with customer_id = 948
import duckdb
con = duckdb.connect()
query = """
SELECT  * from '../data/activity.csv' actibity
WHERE customer_id = 948
Order by from_date
""" 
result = con.execute(query).fetchdf()
display(result)
con.close()

,customer_id,subscription_id,from_date,to_date
0,948,638,2019-02-18,2019-03-18
1,948,18788,2019-08-20,2019-09-19
2,948,18788,2019-09-21,2019-10-23
3,948,22647,2019-10-24,2019-11-19
4,948,34269,2020-05-02,2020-06-01
5,948,34270,2020-05-02,2020-06-01
6,948,36961,2020-06-01,2020-07-01
7,948,36961,2020-07-03,2020-07-31
8,948,36961,2020-08-05,2020-08-27
9,948,492936,2022-08-17,2023-01-23


In [15]:
# Analyse Mental Health and Null business_groups

con = duckdb.connect()
query = """ 
with business_group_activity as (
SELECT 
    ifnull(acq.taxonomy_business_category_group, 'Unknown') as business_group,
    date_trunc('month', activity.from_date) as first_activity_month,
    count(*) as activity_count
FROM '../data/activity.csv' activity
left JOIN '../data/acq_orders.csv' acq on activity.customer_id = acq.customer_id
group by business_group, first_activity_month
ORDER BY first_activity_month)
select * from business_group_activity where business_group in ('Unknown', 'Mental Health Group')
order by business_group, first_activity_month
""" 
result = con.execute(query).fetchdf()
display(result)
con.close() 


,business_group,first_activity_month,activity_count
0,Mental Health Group,2022-07-01,2
1,Mental Health Group,2022-08-01,1
2,Mental Health Group,2022-10-01,1
3,Unknown,2021-02-01,4
4,Unknown,2022-09-01,2
5,Unknown,2023-05-01,1
6,Unknown,2023-08-01,1
7,Unknown,2023-11-01,1
8,Unknown,2024-06-01,1
9,Unknown,2024-08-01,189189


Let's understand better that spike

In [22]:
# Exploring spike in Aug 2024. Query for distribution of groups in in 2024 year to date
con = duckdb.connect()
query = """ 
SELECT 
   taxonomy_business_category_group, date_trunc('month', activity.from_date) as activity_month,
     count(*) as customer_count
FROM '../data/activity.csv' activity
LEFT JOIN '../data/acq_orders.csv' acq ON activity.customer_id = acq.customer_id
WHERE date_trunc('month', activity.from_date) >= '2024-01-01'
GROUP BY taxonomy_business_category_group, activity_month
""" 
result = con.execute(query).fetchdf()

#pivot by month and display results
result_pivot = result.pivot(index='activity_month', columns='taxonomy_business_category_group', values='customer_count').fillna(0) 

display(result_pivot)
con.close()

taxonomy_business_category_group,NaN,ED Group,Hair Loss Group,Other Group,Sleep Group,TRT Group,Weight Loss Group
activity_month,,,,,,,
2024-01-01,0.0,3898.0,62743.0,201.0,62.0,38.0,3636.0
2024-02-01,0.0,3680.0,53778.0,146.0,75.0,59.0,4872.0
2024-03-01,0.0,3497.0,57091.0,205.0,68.0,120.0,5029.0
2024-04-01,0.0,2345.0,57620.0,298.0,73.0,201.0,6135.0
2024-05-01,0.0,2511.0,62605.0,316.0,81.0,309.0,6504.0
2024-06-01,1.0,2221.0,63967.0,277.0,74.0,377.0,6419.0
2024-07-01,0.0,2126.0,63258.0,263.0,58.0,367.0,7205.0
2024-08-01,189189.0,12945.0,186905.0,1995.0,315.0,2895.0,49619.0


In [ ]:
# distribution of business groups in Jul 2024 onwards
con = duckdb.connect()
query = """ 
SELECT 
   taxonomy_business_category_group, activity.from_date,
     count(*) as customer_count
FROM '../data/activity.csv' activity
LEFT JOIN '../data/acq_orders.csv' acq ON activity.customer_id = acq.customer_id
WHERE date_trunc('month', activity.from_date) >= '2024-07-01'
GROUP BY taxonomy_business_category_group, from_date
""" 
result = con.execute(query).fetchdf()

#pivot by month and display results
result_pivot = result.pivot(index='from_date', columns='taxonomy_business_category_group', values='customer_count').fillna(0) 

display(result_pivot)
con.close()

taxonomy_business_category_group,NaN,ED Group,Hair Loss Group,Other Group,Sleep Group,TRT Group,Weight Loss Group
from_date,,,,,,,
2024-07-01,0.0,139.0,2180.0,17.0,9.0,21.0,418.0
2024-07-02,0.0,65.0,2250.0,12.0,0.0,18.0,365.0
2024-07-03,0.0,69.0,2371.0,7.0,0.0,17.0,327.0
2024-07-04,0.0,64.0,2382.0,9.0,2.0,16.0,313.0
2024-07-05,0.0,68.0,2402.0,6.0,2.0,15.0,392.0
2024-07-06,0.0,37.0,1958.0,7.0,0.0,11.0,76.0
2024-07-07,0.0,76.0,2055.0,7.0,3.0,7.0,224.0
2024-07-08,0.0,70.0,3830.0,18.0,6.0,15.0,225.0
2024-07-09,0.0,45.0,3349.0,10.0,0.0,17.0,318.0


It looks like many records with matching from/to dates. how often did that happen in the past?

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# --- Load data ---
activity = pd.read_csv("../data/activity.csv", parse_dates=["from_date", "to_date"])
acq = pd.read_csv("../data/acq_orders.csv")

# --- Filter activity by month >= 2024-01 ---
activity["month"] = activity["from_date"].dt.to_period("M")
filtered = activity[activity["month"] >= pd.Period("2024-01")]

# --- Left join ---
result = filtered.merge(acq, on="customer_id", how="left")
result = result.sort_values(["customer_id", "from_date"])

# --- Flag identical date ranges ---
result["same_dates"] = result["from_date"] == result["to_date"]

# --- Combined table: counts + pct ---
summary = (
    result.groupby("month")
          .agg(
              total_records=("same_dates", "size"),
              same_count=("same_dates", "sum")
          )
          .assign(
              pct_same=lambda df: df["same_count"] / df["total_records"]
          )
          .reset_index()
)
summary

,month,total_records,same_count,pct_same
0,2024-01,70578,804,0.011392
1,2024-02,62610,613,0.009791
2,2024-03,66010,546,0.008271
3,2024-04,66672,692,0.010379
4,2024-05,72326,749,0.010356
5,2024-06,73336,536,0.007309
6,2024-07,73277,848,0.011573
7,2024-08,443863,149588,0.337014
